# Group 4 — IndicLID ensemble: reproduction + fine-tuning

**Runtime > Change runtime type > T4 GPU.** Run top to bottom.

Three tracks:

| Track | What it does | Expected |
|---|---|---|
| A | Published IndicLID ensemble, no training | **80.40 acc / 76.44 Odia F1** (the brief's target) |
| B | Fine-tuned ensemble, held-out benchmark test | higher — this is the saved artifact |
| C | Same artifact on the professor's native Odia set | translationese gap |

**Two landmines already found and fixed below** (do not "simplify" cells 3-4):
1. `torch.load` on the 1.1 GB `.pt` succeeds but the object is unusable — its 2023
   `BertConfig` lacks `_attn_implementation_internal` and blows up on forward.
2. The AI4Bharat repo passes `token_type_ids`; the IndicBERTv2 tokenizer no longer
   returns that key.

Verified working with torch 2.14 / transformers 5.16.


## 1. Setup
> **If you hit `RecursionError` or `ModuleNotFoundError: GenerationMixin`:** your kernel is poisoned from the old cell. **Runtime > Restart session**, then run from the top. Re-running cell 1 is safe in this version.


In [ ]:
!pip -q install fasttext==0.9.3 transformers

# fasttext 0.9.x calls np.array(..., copy=False), which numpy 2.x rejects.
# Scoped shim: only fasttext's view of numpy changes. Patching numpy.array globally
# breaks torch/transformers imports and self-recurses if this cell is re-run.
import types, numpy as np, fasttext.FastText as FT

class _NumpyShim(types.ModuleType):
    def __getattr__(self, name):
        return getattr(np, name)
    @staticmethod
    def array(obj, *args, **kwargs):
        if kwargs.get('copy') is False:
            kwargs['copy'] = None
        return np.array(obj, *args, **kwargs)

FT.np = _NumpyShim('numpy_shim')          # safe to re-run any number of times

import fasttext, torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set runtime to T4')


## 2. Download inputs

SHA-256 values are HuggingFace's published Git-LFS object IDs, so this doubles as an
integrity check.

In [ ]:
import hashlib, os, zipfile

!wget -q -O bhasha.zip https://huggingface.co/datasets/ai4bharat/Bhasha-Abhijnaanam/resolve/main/bhasha-abhijnaanam.zip
!wget -q -O IndicLID-FTR.bin https://huggingface.co/ai4bharat/IndicLID-FTR/resolve/main/model_baseline_roman.bin
!wget -q -O indiclid-bert.pt https://huggingface.co/ai4bharat/IndicLID-BERT/resolve/main/basline_nn_simple.pt

EXPECTED = {
 'bhasha.zip':      'be4bd82c5b9b54528393bcbf4542a4f7f2ac9ee4fc1a3609f0ed685a1d252c19',
 'IndicLID-FTR.bin':'b691f6c44ad1ba04e18d1d8c649ecd3f7f091075600c9d78f3ce5d6c930fa7bf',
 'indiclid-bert.pt':'38462faf42b8b0a5ef66a548abffcf6b7e91f5f5576b54d39a6de96da4a4ce1c',
}
for f, exp in EXPECTED.items():
    h = hashlib.sha256(open(f,'rb').read()).hexdigest()
    print(f'{f:20} {"MATCH" if h==exp else "*** MISMATCH ***"}')

zipfile.ZipFile('bhasha.zip').extractall('.')


## 3. Convert the pickled BERT once

`torch.load` gives back a live 2023 `BertForSequenceClassification`. We take only its
weights and discard the stale object — that is what makes it survive modern
transformers. Note **22 output labels**, not the 47 the repo comments claim: IndicLID-BERT
is the roman-script model only (20 languages + English + Other).

In [ ]:
m_pickle = torch.load('indiclid-bert.pt', map_location='cpu', weights_only=False)
print('num_labels:', m_pickle.config.num_labels, '(EXPECTED 22)')
torch.save(m_pickle.state_dict(), 'indiclid_bert_state_dict.pt')
del m_pickle
print('weights extracted')


## 4. Rebuild the BERT classifier from config + weights

In [ ]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification

TOK = AutoTokenizer.from_pretrained('ai4bharat/IndicBERTv2-MLM-only')

def build_bert(weights='indiclid_bert_state_dict.pt'):
    cfg = AutoConfig.from_pretrained('ai4bharat/IndicBERTv2-MLM-only', num_labels=22)
    model = AutoModelForSequenceClassification.from_config(cfg)
    sd = torch.load(weights, map_location='cpu')
    missing, unexpected = model.load_state_dict(sd, strict=False, assign=True)
    assert not missing, f'missing weights: {missing}'
    print('loaded | unexpected (harmless):', unexpected)
    return model

BERT = build_bert().eval().to('cuda')

# index -> language code, from the AI4Bharat inference code. Verified by probe.
BERT_REV = {0:'asm',1:'ben',2:'brx',3:'guj',4:'hin',5:'kan',6:'kas',7:'kok',8:'mai',
            9:'mal',10:'mni',11:'mar',12:'nep',13:'ori',14:'pan',15:'san',16:'snd',
            17:'tam',18:'tel',19:'urd',20:'eng',21:'other'}


## 5. Splits (seed 42, stratified 80/10/10)

The benchmark is test-only — one flat JSON, no official splits — so we carve our own.
Counts must match: 44649 / 5581 / 5591, with Odia 409 / 51 / 52.

In [ ]:
import json, random, re, collections

SEED, SPLIT = 42, (0.80, 0.10, 0.10)
LANG2CODE = {'Assamese':'asm','Bangla':'ben','Bodo':'brx','Gujarati':'guj','Hindi':'hin',
             'Kannada':'kan','Kashmiri':'kas','Konkani':'kok','Maithili':'mai',
             'Malayalam':'mal','Marathi':'mar','Manipuri':'mni','Nepali':'nep',
             'Oriya':'ori','Punjabi':'pan','Sanskrit':'san','Sindhi':'snd',
             'Tamil':'tam','Telugu':'tel','Urdu':'urd'}
WS = re.compile(r'\s+')
def normalize(t):
    return WS.sub(' ', t.replace('\n',' ')).strip().lower()

d = json.load(open('bhasha-abhijnaanam.json'))['data']
by_lang = collections.defaultdict(list)
for r in d:
    if r.get('language') in LANG2CODE and r.get('romanized sentence','').strip():
        by_lang[r['language']].append({
            'id': r['unique_identifier'],
            'raw': r['romanized sentence'].replace('\n',' '),   # Track A: untouched
            'text': normalize(r['romanized sentence']),          # Track B: normalized
            'label': LANG2CODE[r['language']]})

rng = random.Random(SEED)
splits = {'train': [], 'valid': [], 'test': []}
for lang, items in sorted(by_lang.items()):
    items = sorted(items, key=lambda x: x['id']); rng.shuffle(items)
    n = len(items); n_tr = int(n*SPLIT[0]); n_va = int(n*(SPLIT[0]+SPLIT[1])) - n_tr
    splits['train'] += items[:n_tr]
    splits['valid'] += items[n_tr:n_tr+n_va]
    splits['test']  += items[n_tr+n_va:]
for v in splits.values(): rng.shuffle(v)

print({k: len(v) for k,v in splits.items()}, '(EXPECTED 44649 / 5581 / 5591)')
print('Odia:', {k: sum(1 for i in v if i['label']=='ori') for k,v in splits.items()},
      '(EXPECTED 409 / 51 / 52)')
ALL = [i for v in splits.values() for i in v]


## 6. The ensemble

Faithful to paper §3.4 and the AI4Bharat inference code: route by roman-character
share (>50% roman -> FastText roman model), then escalate to BERT when FastText
confidence is **not** above 0.6.

In [ ]:
SPECIAL = re.compile(r'[@_!#$%^&*()<>?/\\|}{~:]')
EN      = re.compile(r'[a-zA-Z0-9]')

def roman_share(s):
    total = len(s) - len(SPECIAL.findall(s)) - len(re.findall(r'\s', s))
    return 0 if total <= 0 else len(EN.findall(s)) / total

@torch.no_grad()
def bert_predict(texts, batch_size=64, max_length=256):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = TOK(texts[i:i+batch_size], return_tensors='pt', padding=True,
                  truncation=True, max_length=max_length).to('cuda')
        # NOTE: no token_type_ids — the tokenizer no longer emits it.
        logits = BERT(input_ids=enc['input_ids'],
                      attention_mask=enc['attention_mask']).logits
        out += [BERT_REV[i.item()] for i in torch.argmax(logits, 1)]
    return out

def ensemble_predict(texts, ftr, threshold=0.6, use_bert=True, batch_size=64):
    labels, scores = ftr.predict(list(texts))
    preds  = [l[0].replace('__label__','').replace('_Latn','') for l in labels]
    routed = ['FTR'] * len(texts)
    escalate = [i for i, s in enumerate(scores)
                if not (s[0] > threshold) and roman_share(texts[i]) > 0.5]
    if use_bert and escalate:
        print(f'  escalated to BERT: {len(escalate)}/{len(texts)} '
              f'({len(escalate)/len(texts)*100:.1f}%)')
        for i, p in zip(escalate, bert_predict([texts[i] for i in escalate], batch_size)):
            preds[i], routed[i] = p, 'BERT'
    return preds, routed


## Track A — reproduce the published ensemble

**Read this before running.** v1 of this notebook scored 75.25 against a target of
80.40. The cause is now established, and it is not a bug in the routing:

* The FTR binary is byte-identical to both the HuggingFace and GitHub releases.
* The roman-share gate blocks exactly **1** sentence out of 55,821 — it is not filtering
  anything.
* At threshold 0.6 only **12%** of inputs fall below the confidence cut. Even if BERT
  were perfect on all of them, the ceiling is **79.89** — below the target. Hitting 80.40
  at that threshold would require BERT to score 104%.

The released FastText binary produces systematically **higher confidence scores** than the
authors' environment did — most likely a FastText version difference in how
hierarchical-softmax probabilities are computed. Escalation rates implied by the paper's
Table 9 throughputs run ~2.5x ours at every threshold: their 0.6 behaves like our 0.9.

So we run BERT over the whole set **once** and sweep the threshold. This costs one BERT
pass (~10-15 min on a T4) and buys three things: the Table 4 `IndicLID-BERT alone` row
(80.04), the full Table 9 curve, and the ensemble at any threshold for free.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support, classification_report)

def score(y, p, tag, target_acc=None, target_ori=None):
    acc = accuracy_score(y, p) * 100
    mac = f1_score(y, p, average='macro', zero_division=0) * 100
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=['ori'], zero_division=0)
    print(f'\n=== {tag} (n={len(y)}) ===')
    print(f'  accuracy  {acc:6.2f}' + (f'   target {target_acc}' if target_acc else ''))
    print(f'  macro F1  {mac:6.2f}')
    print(f'  Odia F1   {f1[0]*100:6.2f}' + (f'   target {target_ori}' if target_ori else '')
          + f'   (P {pr[0]*100:.2f} / R {rc[0]*100:.2f})')
    return {'accuracy': acc, 'macro_f1': mac, 'odia_f1': f1[0]*100,
            'odia_precision': pr[0]*100, 'odia_recall': rc[0]*100}

FTR_PUB = fasttext.load_model('IndicLID-FTR.bin')
BERT = build_bert().eval().to('cuda')      # published weights, not fine-tuned

y_all   = [r['label'] for r in ALL]
raw_all = [r['raw']   for r in ALL]        # raw: the paper's pre_process is identity

labels, scores = FTR_PUB.predict(raw_all)
conf     = np.array([s[0] for s in scores])
ftr_pred = [l[0].replace('__label__','').replace('_Latn','') for l in labels]

metrics_A_ftr = score(y_all, ftr_pred, 'TRACK A2 - IndicLID-FTR alone', '71.49', '58.26')

print('\nrunning BERT over all', len(raw_all), 'sentences (10-15 min on T4)...')
bert_all = bert_predict(raw_all, batch_size=64, max_length=256)
metrics_A_bert = score(y_all, bert_all, 'TRACK A3 - IndicLID-BERT alone', '80.04', None)

### A4 — threshold sweep (reproduces Table 9)

`paper` holds Table 9's accuracy column. Two numbers get reported downstream: the
ensemble at the paper's literal **0.6**, and the ensemble at whichever threshold actually
reproduces the published behaviour under this binary's calibration.

In [ ]:
paper = {0.1:71.49, 0.2:71.77, 0.3:73.84, 0.4:76.84, 0.5:79.15,
         0.6:80.40, 0.7:80.93, 0.8:80.96, 0.9:80.62}
TARGET_ACC, TARGET_ORI = 80.40, 76.44

sweep = {}
print(f"{'thr':>5}{'%BERT':>8}{'acc':>8}{'OdiaF1':>9}{'paper acc':>11}{'delta':>8}")
for t in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95]:
    esc = conf <= t
    p = [bert_all[i] if esc[i] else ftr_pred[i] for i in range(len(y_all))]
    acc = accuracy_score(y_all, p) * 100
    _, _, f1, _ = precision_recall_fscore_support(y_all, p, labels=['ori'], zero_division=0)
    mac = f1_score(y_all, p, average='macro', zero_division=0) * 100
    sweep[t] = {'accuracy': acc, 'macro_f1': mac, 'odia_f1': f1[0]*100,
                'escalated_pct': esc.mean()*100, 'preds': p}
    ref = paper.get(t)
    print(f'{t:>5.2f}{esc.mean()*100:>8.1f}{acc:>8.2f}{f1[0]*100:>9.2f}'
          f"{(f'{ref:.2f}' if ref else '-'):>11}{(f'{acc-ref:+.2f}' if ref else '-'):>8}")

# literal reproduction at the paper's stated threshold
metrics_A = {k: v for k, v in sweep[0.6].items() if k != 'preds'}

# calibration-matched: threshold whose accuracy lands closest to the published target
best_t = min(sweep, key=lambda t: abs(sweep[t]['accuracy'] - TARGET_ACC))
metrics_A_cal = {k: v for k, v in sweep[best_t].items() if k != 'preds'}
metrics_A_cal['threshold'] = best_t

print(f'\nliteral    threshold 0.6 -> acc {metrics_A["accuracy"]:.2f} '
      f'(target {TARGET_ACC}, delta {metrics_A["accuracy"]-TARGET_ACC:+.2f}), '
      f'Odia F1 {metrics_A["odia_f1"]:.2f} (target {TARGET_ORI}, '
      f'delta {metrics_A["odia_f1"]-TARGET_ORI:+.2f})')
print(f'calibrated threshold {best_t} -> acc {metrics_A_cal["accuracy"]:.2f} '
      f'(delta {metrics_A_cal["accuracy"]-TARGET_ACC:+.2f}), '
      f'Odia F1 {metrics_A_cal["odia_f1"]:.2f} '
      f'(delta {metrics_A_cal["odia_f1"]-TARGET_ORI:+.2f})')

# NOTE: bert_all is kept so this cell can be re-run without redoing the BERT pass.

## Track B — fine-tune both components

**B1: FastText.** Minority classes oversampled to balance; same architecture as the
published FTR (dim 8, char n-grams 3-6, wordNgrams 4).

In [ ]:
def balanced_file(rows, path, seed=SEED):
    by = collections.defaultdict(list)
    for r in rows: by[r['label']].append(r)
    target = max(len(v) for v in by.values())
    rg = random.Random(seed); out = []
    for lab, items in sorted(by.items()):
        out += rg.sample(items, target) if len(items) >= target else \
               items + [rg.choice(items) for _ in range(target - len(items))]
    rg.shuffle(out)
    with open(path, 'w') as f:
        for r in out:
            if r['text'].strip():
                f.write(f"__label__{r['label']} {r['text']}\n")
    return out

# TRAIN ONLY. Earlier versions trained on train+valid, which made validation-based
# selection worthless (the model had memorised valid -> macro F1 100.00). Validation
# is used below to pick both the seed and the ensemble threshold, so it must stay unseen.
train_rows = balanced_file(splits['train'], 'train.ft')
print('balanced train rows:', len(train_rows))

y_valid = [r['label'] for r in splits['valid']]
X_valid = [r['text']  for r in splits['valid']]
y_test  = [r['label'] for r in splits['test']]
X_test  = [r['text']  for r in splits['test']]

def train_ftr(path, seed, lr_ladder=(0.5, 0.3, 0.2, 0.1)):
    last = None
    for lr in lr_ladder:
        try:
            return fasttext.train_supervised(
                path, dim=8, minn=3, maxn=6, wordNgrams=4, lr=lr, epoch=30,
                loss='softmax', bucket=2_000_000, thread=1, seed=seed, verbose=0), lr
        except RuntimeError as e:
            print(f'  seed {seed} lr {lr} diverged - retrying lower'); last = e
    raise last

candidates = []
for s in (42, 7, 1):
    m, lr = train_ftr('train.ft', seed=s)
    pv = [l[0].replace('__label__','') for l in m.predict(X_valid)[0]]
    mf = f1_score(y_valid, pv, average='macro', zero_division=0) * 100
    print(f'seed {s:>3} lr {lr}: valid macroF1 {mf:6.2f}   <- must NOT be 100')
    candidates.append((mf, s, lr, m))

candidates.sort(key=lambda c: -c[0])
FTR_VALID_MF1, FTR_SEED, FTR_LR, FTR_FT = candidates[0]
print(f'\nselected seed {FTR_SEED} (lr {FTR_LR}), valid macro F1 {FTR_VALID_MF1:.2f}')
print(f'seed spread: {max(c[0] for c in candidates)-min(c[0] for c in candidates):.2f} macro F1 points')
FTR_FT.save_model('indiclid_ftr_finetuned.bin')

p_ft, _ = ensemble_predict(X_test, FTR_FT, use_bert=False)
metrics_B1 = score(y_test, p_ft, 'TRACK B1 - fine-tuned FastText alone')


**B2: IndicBERT.** Paper's `unfreeze-layer-1` setting — only the final encoder
layer, pooler and classifier head train. Roughly 25-40 min on a T4 for one epoch. Drop
`EPOCHS` to 1 if you are short on time; the marginal gain past epoch 1 is small.

In [ ]:
from torch.utils.data import Dataset, DataLoader

EPOCHS, BATCH, LR = 1, 32, 2e-5
CODE2IDX = {v: k for k, v in BERT_REV.items()}

class DS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]['text'], CODE2IDX[self.rows[i]['label']]

def collate(batch):
    texts, labels = zip(*batch)
    enc = TOK(list(texts), return_tensors='pt', padding=True, truncation=True, max_length=128)
    return enc, torch.tensor(labels)

BERT_FT = build_bert().to('cuda')
for p in BERT_FT.parameters(): p.requires_grad = False
for module in [BERT_FT.bert.encoder.layer[-1], BERT_FT.bert.pooler, BERT_FT.classifier]:
    for p in module.parameters(): p.requires_grad = True
trainable = sum(p.numel() for p in BERT_FT.parameters() if p.requires_grad)
print(f'trainable params: {trainable/1e6:.1f}M of {sum(p.numel() for p in BERT_FT.parameters())/1e6:.1f}M')

opt = torch.optim.AdamW([p for p in BERT_FT.parameters() if p.requires_grad], lr=LR)
lossf = torch.nn.CrossEntropyLoss()
dl = DataLoader(DS(train_rows), batch_size=BATCH, shuffle=True, collate_fn=collate)

BERT_FT.train()
for ep in range(EPOCHS):
    running = 0.0
    for step, (enc, lab) in enumerate(dl):
        enc, lab = {k: v.to('cuda') for k, v in enc.items()}, lab.to('cuda')
        out = BERT_FT(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
        loss = lossf(out.logits, lab)
        loss.backward(); opt.step(); opt.zero_grad()
        running += loss.item()
        if step % 200 == 0:
            print(f'  epoch {ep} step {step}/{len(dl)} loss {running/(step+1):.4f}', flush=True)
BERT_FT.eval()
torch.save(BERT_FT.state_dict(), 'indiclid_bert_finetuned.pt')
print('saved indiclid_bert_finetuned.pt')


**B3: fine-tuned ensemble on the held-out benchmark test.**

This is the artifact that carries forward to Track C.

In [ ]:
BERT = BERT_FT          # ensemble_predict reads the global BERT

# Track A established that this fasttext build emits inflated confidences, so 0.6
# escalates almost nothing (1/5591 last run). Pick the threshold on VALIDATION.
conf_v = np.array([s[0] for s in FTR_FT.predict(X_valid)[1]])
ftr_v  = [l[0].replace('__label__','') for l in FTR_FT.predict(X_valid)[0]]
bert_v = bert_predict(X_valid, batch_size=64, max_length=256)

print(f"{'thr':>6}{'%BERT':>8}{'valid macroF1':>15}")
best = (-1, 0.6)
for t in [0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.01]:
    esc = conf_v <= t
    p = [bert_v[i] if esc[i] else ftr_v[i] for i in range(len(y_valid))]
    mf = f1_score(y_valid, p, average='macro', zero_division=0) * 100
    print(f'{t:>6.2f}{esc.mean()*100:>8.1f}{mf:>15.2f}')
    if mf > best[0]: best = (mf, t)
B_THRESHOLD = best[1]
print(f'\nselected ensemble threshold {B_THRESHOLD} (valid macro F1 {best[0]:.2f})')

conf_t = np.array([s[0] for s in FTR_FT.predict(X_test)[1]])
ftr_t  = [l[0].replace('__label__','') for l in FTR_FT.predict(X_test)[0]]
bert_t = bert_predict(X_test, batch_size=64, max_length=256)
esc_t  = conf_t <= B_THRESHOLD
p_ens  = [bert_t[i] if esc_t[i] else ftr_t[i] for i in range(len(y_test))]
print(f'escalated on test: {esc_t.sum()}/{len(y_test)} ({esc_t.mean()*100:.1f}%)')

metrics_B = score(y_test, p_ens, 'TRACK B - fine-tuned ensemble (held-out test)')
metrics_B['threshold'] = B_THRESHOLD
print(classification_report(y_test, p_ens, zero_division=0, digits=3))

with open('predictions_benchmark_test.jsonl','w') as f:
    for r, p in zip(splits['test'], p_ens):
        f.write(json.dumps({'id': r['id'], 'gold': r['label'], 'pred': p},
                           ensure_ascii=False) + '\n')


## Track C — native Odia gold set

Upload the professor's file, then adapt the parser to its actual columns. **Same model,
same preprocessing, same metric** — that is the whole point of the step, so do not retune
anything here.

Two things to check the moment you open the file:
- **Script.** If it is Odia script rather than romanized, the roman-share router sends it
  to the native model and this pipeline is the wrong one — raise it immediately.
- **Labels.** If every row is Odia, F1 collapses to recall. You then need a negative set,
  and you must say so explicitly in the report.

In [ ]:
from google.colab import files
up = files.upload()
gold_path = list(up.keys())[0]
print('uploaded:', gold_path)

# --- ADAPT THIS to the file's real structure ---------------------------------
import pandas as pd
gold = pd.read_csv(gold_path)          # or read_excel / read_json / plain text
print(gold.head())
print('\nrows:', len(gold), '| columns:', list(gold.columns))
print('label distribution:\n', gold.iloc[:, -1].value_counts())


In [ ]:
TEXT_COL, LABEL_COL = 'text', 'label'      # <-- set these from the output above

gold_texts  = [normalize(t) for t in gold[TEXT_COL].astype(str)]
gold_labels = list(gold[LABEL_COL])                # must use the same codes ('ori', ...)

print('roman share (mean):', np.mean([roman_share(t) for t in gold_texts]).round(3),
      '-> >0.5 means romanized, which is what this pipeline expects')

p_gold, routed_C = ensemble_predict(gold_texts, FTR_FT, use_bert=True,
                                    threshold=B_THRESHOLD)
metrics_C = score(gold_labels, p_gold, 'TRACK C — native Odia gold set')
print('routing:', collections.Counter(routed_C))


## Final results table

## Uncertainty — bootstrap confidence intervals

Odia is under 1% of the benchmark, so Odia F1 rests on 512 rows in Track A and only **52**
in the Track B held-out test. At n=52 its 95% CI is roughly **10 points wide**, versus ~1
point for accuracy. Any Odia F1 quoted without an interval is over-claiming, and a
translationese gap computed from two small samples inherits both intervals.

This runs on predictions already in memory — no retraining, no second BERT pass.


In [ ]:
def _fast(y, p, ori='ori'):
    acc = (y == p).mean() * 100
    tp = ((y == ori) & (p == ori)).sum()
    fp = ((y != ori) & (p == ori)).sum()
    fn = ((y == ori) & (p != ori)).sum()
    pr = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
    return acc, f1 * 100

def boot_ci(y, p, B=2000, seed=0, macro_B=300):
    y = np.asarray(y); p = np.asarray(p); n = len(y)
    rng = np.random.default_rng(seed)
    accs = np.empty(B); oris = np.empty(B)
    for b in range(B):
        i = rng.integers(0, n, n); accs[b], oris[b] = _fast(y[i], p[i])
    macs = np.empty(macro_B)
    for b in range(macro_B):
        i = rng.integers(0, n, n)
        macs[b] = f1_score(y[i], p[i], average='macro', zero_division=0) * 100
    q = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return {'n': n, 'n_odia': int((y == 'ori').sum()),
            'accuracy_ci': q(accs), 'macro_f1_ci': q(macs), 'odia_f1_ci': q(oris),
            '_acc_boot': accs, '_odia_boot': oris}

CI = {}
G = globals()
if 'y_all' in G and 'ftr_pred' in G:  CI['A2 FTR alone']      = boot_ci(y_all, ftr_pred, B=1000)
if 'y_all' in G and 'bert_all' in G:  CI['A3 BERT alone']     = boot_ci(y_all, bert_all, B=1000)
if 'sweep' in G and metrics_A_cal['threshold'] in sweep:
    CI['A ensemble (cal.)'] = boot_ci(y_all, sweep[metrics_A_cal['threshold']]['preds'], B=1000)
if 'p_ens' in G:      CI['B fine-tuned ensemble'] = boot_ci(y_test, p_ens)
if 'p_gold' in G:     CI['C native Odia']         = boot_ci(gold_labels, p_gold)

print(f"{'':26}{'n':>7}{'nOdia':>7}{'accuracy 95% CI':>24}{'Odia F1 95% CI':>24}")
for k, v in CI.items():
    a, o = v['accuracy_ci'], v['odia_f1_ci']
    print(f"{k:26}{v['n']:>7}{v['n_odia']:>7}"
          f"{f'[{a[0]:.2f}, {a[1]:.2f}] w{a[1]-a[0]:.2f}':>24}"
          f"{f'[{o[0]:.2f}, {o[1]:.2f}] w{o[1]-o[0]:.2f}':>24}")

GAP_CI = None
if 'B fine-tuned ensemble' in CI and 'C native Odia' in CI:
    b, c = CI['B fine-tuned ensemble'], CI['C native Odia']
    m = min(len(b['_acc_boot']), len(c['_acc_boot']))
    ga = b['_acc_boot'][:m] - c['_acc_boot'][:m]
    go = b['_odia_boot'][:m] - c['_odia_boot'][:m]
    GAP_CI = {'accuracy': (float(np.percentile(ga, 2.5)), float(np.percentile(ga, 97.5))),
              'odia_f1':  (float(np.percentile(go, 2.5)), float(np.percentile(go, 97.5)))}
    print(f"\nTRANSLATIONESE GAP 95% CI")
    print(f"  accuracy [{GAP_CI['accuracy'][0]:.2f}, {GAP_CI['accuracy'][1]:.2f}]")
    print(f"  Odia F1  [{GAP_CI['odia_f1'][0]:.2f}, {GAP_CI['odia_f1'][1]:.2f}]")
    if GAP_CI['odia_f1'][0] < 0 < GAP_CI['odia_f1'][1]:
        print('  WARNING: Odia F1 gap CI spans zero - not statistically distinguishable.')
        print('           Report the accuracy gap as primary and state this explicitly.')

CI_CLEAN = {k: {kk: vv for kk, vv in v.items() if not kk.startswith('_')}
            for k, v in CI.items()}


In [ ]:
HAVE_C = 'metrics_C' in globals()
if not HAVE_C:
    print('Track C not run yet - native Odia rows and the gap are omitted.\n')
    metrics_C = None
gap_acc = metrics_B['accuracy'] - metrics_C['accuracy'] if HAVE_C else None
gap_f1  = metrics_B['odia_f1']  - metrics_C['odia_f1']  if HAVE_C else None

rows = [
 ('Benchmark - FTR alone',            'accuracy', '71.49', metrics_A_ftr['accuracy']),
 ('Benchmark - FTR alone',            'Odia F1',  '58.26', metrics_A_ftr['odia_f1']),
 ('Benchmark - BERT alone',           'accuracy', '80.04', metrics_A_bert['accuracy']),
 ('Benchmark - ensemble @0.6',        'accuracy', '80.40', metrics_A['accuracy']),
 ('Benchmark - ensemble @0.6',        'Odia F1',  '76.44', metrics_A['odia_f1']),
 (f"Benchmark - ensemble @{metrics_A_cal['threshold']} (cal.)",
                                      'accuracy', '80.40', metrics_A_cal['accuracy']),
 (f"Benchmark - ensemble @{metrics_A_cal['threshold']} (cal.)",
                                      'Odia F1',  '76.44', metrics_A_cal['odia_f1']),
 ('Benchmark test - fine-tuned',      'accuracy', '-',     metrics_B['accuracy']),
 ('Benchmark test - fine-tuned',      'Odia F1',  '-',     metrics_B['odia_f1']),
]
if HAVE_C:
    rows += [('Native Odia gold', 'accuracy', 'N/A', metrics_C['accuracy']),
             ('Native Odia gold', 'Odia F1',  'N/A', metrics_C['odia_f1'])]
print(f"{'Evaluation Dataset':<40}{'Metric':<10}{'Target':>9}{'Ours':>9}{'Diff':>9}")
print('-' * 77)
for name, metric, target, ours in rows:
    diff = f'{ours-float(target):+.2f}' if target not in ('-', 'N/A') else '-'
    print(f'{name:<40}{metric:<10}{target:>9}{ours:>9.2f}{diff:>9}')

if HAVE_C:
    print(f'\nTRANSLATIONESE GAP (accuracy): {gap_acc:.2f}')
    print(f'TRANSLATIONESE GAP (Odia F1):  {gap_f1:.2f}')

json.dump({'track_A2_ftr_alone': metrics_A_ftr,
           'track_A3_bert_alone': metrics_A_bert,
           'track_A_ensemble_at_0.6': metrics_A,
           'track_A_ensemble_calibrated': metrics_A_cal,
           'track_A4_threshold_sweep': {str(k): {kk: vv for kk, vv in v.items()
                                                 if kk != 'preds'}
                                        for k, v in sweep.items()},
           'track_B1_finetuned_ftr_alone': metrics_B1,
           'track_B_finetuned_ensemble': metrics_B,
           'track_C_native_odia': metrics_C if HAVE_C else None,
           'translationese_gap': ({'accuracy': gap_acc, 'odia_f1': gap_f1} if HAVE_C else None),
           'confidence_intervals': CI_CLEAN,
           'translationese_gap_ci': GAP_CI,
           'config': {'seed': SEED, 'epochs': EPOCHS, 'batch': BATCH, 'lr': LR,
                      'fasttext_lr': FTR_LR, 'fasttext_thread': 1,
                      'fasttext_seed': FTR_SEED,
                      'fasttext_valid_macro_f1': FTR_VALID_MF1,
                      'ensemble_threshold': B_THRESHOLD,
                      'trained_on': 'train split only'}},
          open('metrics_all.json', 'w'), indent=2)

files.download('metrics_all.json')
files.download('indiclid_ftr_finetuned.bin')
files.download('predictions_benchmark_test.jsonl')